# Análise Exploratória de Dados (EDA) - INCT SIMAI
**Projeto Extensionista - Fase 2**

Este notebook contém a Análise Exploratória de Dados (EDA) estruturada a partir da base `data_atualizado.json` limpa (com os tratamentos apontados no relatório de limpeza para os bairros Arquipélago, Espírito Santo, Lageado, Centro Histórico e Rubem Berta). 
Aqui temos gráficos comentados que farão parte do escopo da **Tela 1** (Simulação do INCT) e explorações para a **Tela 2** (Bases Externas). 
Ao final, é apresentado o esboço (wireframe) detalhado das duas telas.


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json
import numpy as np

# Configuração de tema padrão
import plotly.io as pio
pio.templates.default = "plotly_white"


In [2]:
# ==========================================
# PREPARAÇÃO DOS DADOS DA TELA 1 (JSON INCT)
# ==========================================
# Carregamento do JSON limpo.

with open("../dados/processados/data_atualizado.json", encoding="utf-8") as f:
        raw = json.load(f)
        
df = pd.DataFrame([
    {"nivel_m": float(nivel.removesuffix("m").replace("_", ".")), 
        "bairro": bairro, **valores}
    for nivel, bairros in raw.items()
    for bairro, valores in bairros.items()
])

display(df.head())
print(f"Formato dos dados: {df.shape}")


,nivel_m,bairro,affected_buildings,total_buildings,percentage,population,affected_population
0,3.0,Aberta dos Morros,0,4696,0,9854,0
1,3.0,Agronomia,0,1771,0,2677,0
2,3.0,Anchieta,28,589,5,791,40
3,3.0,Arquipélago,4688,4688,100,6411,6411
4,3.0,Auxiliadora,0,1995,0,8909,0


Formato dos dados: (6674, 7)


### Gráfico 1 (Tela 1): Curva da População Afetada por Nível
**Requisito:** Uma curva da população afetada em toda a cidade por nível do rio, com o nível recorde de maio de 2024 (5,35 m) marcado.


In [ ]:
df_populacao_total = df.groupby('nivel_m')['affected_population'].sum().reset_index()

fig1 = px.line(df_populacao_total, x='nivel_m', y='affected_population', 
               title='População Afetada vs Nível do Rio Guaíba',
               labels={'nivel_m': 'Nível do Rio (m)', 'affected_population': 'População Total Afetada'})

# Marcação do Recorde Histórico de 2024
fig1.add_vline(x=5.35, line_dash="dash", line_color="red", 
               annotation_text="Recorde Maio 2024 (5,35m)")

fig1.show()

# Este gráfico demonstra a curva de crescimento do impacto populacional. 
# A marcação em 5,35m evidencia o ponto de catástrofe com tamanho maior.


### Gráfico 2 (Tela 1): Porcentagem de Prédios Afetados por Bairro
**Requisito:** Um gráfico com a porcentagem de prédios afetados em cada bairro atingido (simulando um filtro fixo para 5,35m).


In [ ]:
nivel_selecionado = 5.35
df_nivel = df.iloc[(df['nivel_m'] - nivel_selecionado).abs().argsort()[:len(df['bairro'].unique())]]
df_nivel = df_nivel[df_nivel['percentage'] > 0].sort_values('percentage', ascending=False)

fig2 = px.bar(df_nivel, x='bairro', y='percentage', 
              title=f'Porcentagem de Prédios Afetados por Bairro (Nível: {nivel_selecionado}m)',
              labels={'percentage': '% de Prédios Afetados', 'bairro': 'Bairro'},
              color='percentage', color_continuous_scale='Reds')

fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

# Este gráfico exibe a proporção de danos na infraestrutura de cada localidade atingida. 
# O gradiente de cor reforça a intensidade do impacto e orienta onde a mancha está mais agressiva.


### Gráfico 3 (Tela 1): Ranking dos Bairros Mais Afetados
**Requisito:** Um ranking dos bairros mais afetados no nível escolhido.


In [ ]:
df_ranking = df_nivel.sort_values('affected_buildings', ascending=False).head(10)

fig3 = px.bar(df_ranking, y='bairro', x='affected_buildings', orientation='h',
              title=f'Top 10 Bairros Mais Afetados em volume absoluto (Nível: {nivel_selecionado}m)',
              labels={'affected_buildings': 'Prédios Afetados', 'bairro': 'Bairro'})

fig3.update_layout(yaxis={'categoryorder':'total ascending'})
fig3.show()

# Visualização em barras horizontais facilita a leitura dos nomes dos bairros. 
# Diferente do percentual, aqui vemos o volume absoluto de prédios debaixo d'água.


### Gráfico 4 (Tela 1): Avanço da Mancha de Inundação (Apoio aos Cards)
Visão complementar para entender como a quantidade de bairros atingidos sobe com a elevação.


In [ ]:
df_bairros_atingidos = df[df['affected_buildings'] > 0].groupby('nivel_m')['bairro'].count().reset_index()

fig4 = px.area(df_bairros_atingidos, x='nivel_m', y='bairro',
               title='Quantidade de Bairros Atingidos conforme Elevação do Rio',
               labels={'bairro': 'Qtd. de Bairros Atingidos', 'nivel_m': 'Nível do Rio (m)'})

fig4.add_vline(x=5.35, line_dash="dash", line_color="red")
fig4.show()

# Gráfico de área para dar suporte aos indicadores do topo da Tela 1, 
# ilustrando a velocidade com que a enchente se espalha geograficamente.


### Preparação - Bases Externas (Tela 2)
Carregamento das bases da Pesquisa PEERS (IBGE) para a Região Intermediária de Porto Alegre, focando nas consequências das enchentes de maio de 2024.


In [7]:
# Tabela 22.4 (Tipos de Ocorrências no Domicílio)
df_ocorrencias = pd.read_excel('../dados/originais/Tabela22.4.xlsx', skiprows=4)
df_ocorrencias.columns = ['Tipo', 'Total', 'Percentual']
df_ocorrencias = df_ocorrencias.iloc[0:8].copy() # Pegando apenas Ocorrências no Domicílio
df_ocorrencias.dropna(subset=['Percentual'], inplace=True)

# Tabela 22.6 (Avaliação da Qualidade de Vida)
df_qualidade = pd.read_excel('../dados/originais/Tabela22.6.xlsx', skiprows=4)
df_qualidade.columns = ['Categoria', 'Total', 'Percentual']
df_vida_geral = df_qualidade.iloc[1:4].copy() # Avaliação da vida em geral

display(df_ocorrencias.head(3))
display(df_vida_geral.head(3))


,Tipo,Total,Percentual
0,Domicílio impossibilitado de ser acessado,453757.427730,32.853276
1,Domicílio inundado,369180.980985,26.729710
2,"Danos estruturais (infiltração, mofo, paredes,...",728691.321672,52.759240


,Categoria,Total,Percentual
1,Pior que antes das enchentes,9.935237e+05,26.592478
2,Igual a antes das enchentes,2.018165e+06,54.017852
3,NaN,NaN,NaN


### Gráfico 5 (Tela 2): Principais Ocorrências nos Domicílios
**Requisito:** Mostrar os dados externos levantados (PEERS/IBGE) e sua distribuição.


In [ ]:
df_ocorrencias['Tipo'] = df_ocorrencias['Tipo'].str.wrap(40)
df_ocorrencias = df_ocorrencias.sort_values('Percentual', ascending=True)

fig5 = px.bar(df_ocorrencias, x='Percentual', y='Tipo', orientation='h',
              title='Impactos Reportados nos Domicílios (Região de Porto Alegre - PEERS/IBGE)',
              labels={'Percentual': '% de Domicílios Afetados', 'Tipo': ''},
              color='Percentual', color_continuous_scale='Blues')

fig5.show()

# Este gráfico expõe o impacto real reportado pelos moradores (IBGE). 
# Ele complementa a simulação da Tela 1 mostrando que a falta de água e luz afetou muito mais gente do que a inundação direta.


### Gráfico 6 (Tela 2): Percepção de Piora na Qualidade de Vida
Avaliação geral após as enchentes.


In [ ]:
fig6 = px.pie(df_vida_geral, values='Percentual', names='Categoria', 
              title='Avaliação da Vida em Geral após as Enchentes (PEERS/IBGE)',
              color='Categoria',
              color_discrete_map={'Pior que antes das enchentes':'#EF553B', 
                                  'Igual a antes das enchentes':'#636efa', 
                                  'Melhor que antes das enchentes':'#00cc96'})
fig6.update_traces(textposition='inside', textinfo='percent+label')
fig6.show()

# Uma visão subjetiva do impacto. Mais de 25% reportam piora na qualidade de vida,
# conectando os dados de infraestrutura com o impacto humano e social.


## Esboço (Wireframe) das Telas

### Tela 1 - Simulação do INCT

Abaixo está a proposta visual da primeira tela do dashboard.

![Wireframe Tela 1](Wireframe_Tela_1_INCT.png)

### Tela 2 - Análise Contextual & Externa

Abaixo está a proposta visual da segunda tela do dashboard.

![Wireframe Tela 2](Wireframe_Tela_2.png)